# Lab 3 — The Cost–Accuracy Frontier

**MA553 · Session 4**

Name:

---

A taxi company wants to tell each passenger, at pickup, how many minutes the
trip will take. You will build that prediction with the five model families
from the session, measure what each one costs, and choose a model for two
different uses.

**Part A** fits the five families and measures three things for each: the time
to fit, the time to predict, and the test error.

**Part B** audits the linear model, builds the frontier table, and asks you to
choose a model under two budgets.

Work through the cells in order. Five cells are marked **Write this yourself**:
A3, A4, B1, B2 and B3. Each says which variables to create; later cells use
those names. B4 and B5 are short written answers.

# Part A — Fitting the five families

## The data

This lab uses the January 2025 file you downloaded for Lab 2. The cell below
looks for it in `../lab2/data/`. If it is missing, run `python get_data.py` in
the `lab2` folder first.

Each row is one trip. The columns used here:

| Column | Meaning |
|---|---|
| `tpep_pickup_datetime`, `tpep_dropoff_datetime` | when the trip began and ended |
| `trip_distance` | miles, from the meter |
| `PULocationID`, `DOLocationID` | pickup and dropoff zone numbers |

Run this.

In [ ]:
import pickle
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor, plot_tree

PATH = Path("../lab2/data/yellow_tripdata_2025-01.parquet")
COLUMNS = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance",
           "PULocationID", "DOLocationID"]
trips = pd.read_parquet(PATH, columns=COLUMNS)
print(f"Trips in the file: {len(trips):,}")

## A1 — The task, the test set and the baseline

The **target** is the trip duration in minutes. The five **features** are known
at pickup: distance, pickup hour, weekday, pickup zone and dropoff zone.

The cell keeps trips of 1 to 120 minutes with a distance above 0 and below 60
miles. It then sets aside 100,000 trips once, as the **test set**. Every error
in this lab is measured on these trips. The remaining trips are the pool from
which training samples are drawn.

Run this.

In [ ]:
trips["minutes"] = (trips["tpep_dropoff_datetime"] - trips["tpep_pickup_datetime"]).dt.total_seconds() / 60
keep = trips["minutes"].between(1, 120) & (trips["trip_distance"] > 0) & (trips["trip_distance"] < 60)
trips = trips[keep].copy()
trips["hour"] = trips["tpep_pickup_datetime"].dt.hour
trips["weekday"] = trips["tpep_pickup_datetime"].dt.weekday

FEATURES = ["trip_distance", "hour", "weekday", "PULocationID", "DOLocationID"]

test = trips.sample(100_000, random_state=1)
pool = trips.drop(test.index)


def training_sample(n):
    # The same n trips every time this is called with the same n.
    return pool.sample(n, random_state=2)


baseline = np.full(len(test), pool["minutes"].median())
baseline_mae = mean_absolute_error(test["minutes"], baseline)

print(f"Trips kept:          {len(trips):,}")
print(f"Test set:            {len(test):,}")
print(f"Pool for training:   {len(pool):,}")
print(f"Baseline: predict {pool['minutes'].median():.1f} minutes for every trip. Test MAE {baseline_mae:.2f} minutes.")

Expected: 3,368,501 trips kept and a baseline MAE of 7.39 minutes.

## A2 — One family from start to finish: the linear model

This cell fits the linear model (ridge regression) to 10,000 training trips,
exactly as in the slides. Distance, hour and weekday are rescaled. The two
zone columns are one-hot encoded.

It measures four things:

- **fit time**, in seconds;
- **predict time** per trip, in microseconds (millionths of a second), from
  predicting the 100,000 test trips;
- **test MAE**, in minutes;
- **model size**, in megabytes, the memory needed to store the fitted model.

The function `measure` does these four measurements for any model. Later cells
use it. Run this.

In [ ]:
def measure(name, model, n, columns=FEATURES):
    train = training_sample(n)
    start = time.time()
    model.fit(train[columns], train["minutes"])
    fit_s = time.time() - start

    start = time.time()
    predicted = model.predict(test[columns])
    predict_us = (time.time() - start) / len(test) * 1e6

    return {"model": name, "trips": n, "fit_s": round(fit_s, 2),
            "predict_us_per_trip": round(predict_us, 2),
            "mae": round(mean_absolute_error(test["minutes"], predicted), 2),
            "size_mb": round(len(pickle.dumps(model)) / 1e6, 2)}


def linear_model():
    prepare = ColumnTransformer([
        ("numbers", StandardScaler(), ["trip_distance", "hour", "weekday"]),
        ("zones", OneHotEncoder(handle_unknown="ignore"), ["PULocationID", "DOLocationID"]),
    ])
    return make_pipeline(prepare, Ridge(alpha=1.0))


row = measure("linear", linear_model(), 10_000)
print(row)

Expected: a test MAE of 4.27 minutes. The times depend on your machine.

## A3 — The other four families

**Write this yourself.**

Fit the other four families to the same 10,000 training trips and collect the
results in one table, together with the linear model's row from A2.

What to do:

1. Create the four models with these settings, the same as in the slides:
   - k-nearest neighbours: the same preparation as `linear_model()`, followed
     by `KNeighborsRegressor(n_neighbors=20, n_jobs=-1)`;
   - decision tree: `DecisionTreeRegressor(max_depth=12, random_state=0)`;
   - random forest: `RandomForestRegressor(n_estimators=100, max_depth=12, n_jobs=-1, random_state=0)`;
   - gradient boosting: `HistGradientBoostingRegressor(max_iter=300, random_state=0)`.

   The three tree-based models take the five features directly; they need no
   preparation step.
2. Call `measure` for each one with `n = 10_000`.
3. Store the five rows (including `row` from A2) in a DataFrame named
   `results_a3` and print it.

Expected test MAE: tree 4.02, forest 3.37, boosting 3.19, nearest neighbours
4.15. Nearest neighbours takes about 20 seconds to predict.

In [ ]:
# Your code here.


**Answer here.** Which family has the lowest test MAE, and which has the
highest predict time per trip? Give both numbers.

*Your answer:*

## A4 — A learning curve

**Write this yourself.**

A learning curve shows how the test error changes as the number of training
trips grows.

What to do:

1. Choose one family other than nearest neighbours.
2. Measure it at 10,000, 100,000 and 1,000,000 training trips with `measure`.
   Store the three rows in a DataFrame named `curve`.
3. Plot the test MAE against the number of training trips, with a log scale
   on the horizontal axis (`plt.xscale("log")`). Label both axes.

The random forest takes one to two minutes at 1,000,000 trips. Run it once.

In [ ]:
# Your code here.


**Answer here.** By how many minutes did the test MAE fall from 10,000 to
1,000,000 trips? Using the terms bias and variance, say what this tells you
about the family you chose.

*Your answer:*

## A5 — Reading a tree

This cell fits a tree of depth 3 to 100,000 training trips and draws it. Each
box shows its question, the number of training trips that reached it
(`samples`), and their average duration (`value`). Run this.

In [ ]:
small_tree = DecisionTreeRegressor(max_depth=3, random_state=0)
train = training_sample(100_000)
small_tree.fit(train[FEATURES], train["minutes"])

plt.figure(figsize=(16, 6))
plot_tree(small_tree, feature_names=["distance", "hour", "weekday", "pickup zone", "dropoff zone"],
          impurity=False, precision=1, fontsize=9)
plt.show()

**Answer here.**

1. Follow a 2-mile trip from the top of the tree to its leaf. What duration
   does the tree predict?
2. How many different predictions can this tree give in total, and why?

*Your answer:*

# Part B — Auditing the result and choosing a model

## B1 — Audit the linear model

**Write this yourself.**

Look again at how `linear_model()` in A2 passes the pickup hour to the ridge
regression. It receives the hour as one number from 0 to 23, and it learns one
coefficient for it. The hour effect it can represent is therefore a straight
line. This cell checks whether a straight line fits how duration varies with
the hour.

What to do:

1. Fit `linear_model()` to 1,000,000 training trips. Call it `linear_1m`.
2. Compute the average duration for each pickup hour in those training trips,
   minus the overall average. Store it as `hour_pattern`, a Series indexed by
   hour.
3. Compute the hour effect the model learned, in minutes, for each hour 0–23:
   the coefficient times the rescaled hour. The coefficient is
   `linear_1m[-1].coef_[1]`. The rescaling uses the mean and standard deviation
   stored in `linear_1m[0].named_transformers_["numbers"]` (`.mean_[1]` and
   `.scale_[1]`). Subtract its average over the 24 hours. Store it as
   `hour_effect`.
4. Plot both against the hour on one figure, with a legend.
5. Write a corrected model, `linear_hour_onehot()`, in which hour and weekday
   are one-hot encoded like the zones, and only distance is rescaled. Measure
   it at 1,000,000 trips with `measure` and store the row as `row_hour_onehot`.

The average by hour also reflects that trip distances differ by hour, so the
two curves need not match exactly. Compare their shapes.

Expected: the test MAE at 1,000,000 trips falls from 4.12 to 3.95 minutes.

In [ ]:
# Your code here.


**Answer here.** Describe the shape of the average duration by hour in one
sentence. Explain why the model in A2 could not represent it, and why one-hot
encoding the hour can.

*Your answer:*

## B2 — Add combinations of zones

**Write this yourself.**

The slides showed that the linear model adds one term for the pickup zone and
one for the dropoff zone, so it cannot represent a route that is slower or
faster than the sum of its two zone terms. Give it a term for each route.

What to do:

1. Add a column `pair` to both `pool` and `test`: the pickup and dropoff zone
   numbers joined as text, for example `"161-237"`.
2. Write `linear_pairs()`: the model from B1, plus one-hot encoding of `pair`
   with `OneHotEncoder(handle_unknown="ignore", min_frequency=5)`. Pairs seen
   fewer than 5 times in the training trips are grouped together.
3. Measure it at 10,000, 100,000 and 1,000,000 trips, passing
   `columns=FEATURES + ["pair"]` to `measure`. Store the three rows in a
   DataFrame named `pairs_results` and print it.

Expected test MAE: 3.94, 3.45 and 3.36 minutes.

In [ ]:
# Your code here.


**Answer here.** In the slides, 100 times more data reduced the linear model's
error by only 0.15 minutes. With the pair columns, more data helps much more.
Explain why, using the terms bias and variance.

*Your answer:*

## B3 — The frontier table

**Write this yourself.**

Build one table that compares every model at 1,000,000 training trips.

What to do:

1. Measure the decision tree, the random forest and gradient boosting at
   1,000,000 trips with `measure`. Reuse a row from A4 if you already measured
   that family at 1,000,000 trips.
2. Combine them with `row_hour_numeric` and `row_hour_onehot` from B1, the
   1,000,000-trip row of `pairs_results` from B2, and the nearest-neighbours
   row from A3 (measured at 10,000 trips; it is too slow at larger sizes).
3. Store the table as `frontier`, sorted by test MAE, and print it.

The random forest takes one to two minutes at 1,000,000 trips.

In [ ]:
# Your code here.


## B4 — Choosing a model under two budgets

Answer each part using the numbers in your `frontier` table. For each budget,
name the model you choose, say which models fail the budget and on which
requirement, and give the test MAE of your choice.

**Budget 1: a dispatch server.** The model is refit every week on the latest
1,000,000 trips, and the refit must take under 60 seconds. Each prediction
must take under 1 millisecond (1,000 microseconds).

*Your answer:*

**Budget 2: a published estimate.** The city requires that every predicted
duration be explained as a sum of per-feature contributions in minutes,
published as a table. For example: "pickup hour 17: +2.1 minutes". Which
families can meet this requirement? Which model do you choose? How many
minutes of test MAE does the requirement cost, compared with your choice under
Budget 1?

*Your answer:*

## B5 — One limitation

Name one thing your choices in B4 do not establish, and what would be needed
to establish it.

*Your answer:*

## Before you submit

- Restart the kernel and run every cell from top to bottom. A4 and B3 take a
  few minutes; that is expected.
- Check that every **Answer here** cell and both parts of B4 are filled in.
- Do not submit the data file.
- If you used an AI assistant, add a short note at the end saying what it
  produced and what you changed. You must be able to explain every line.